# Rx PD Model - Colab Ready
## Lending Default Prediction Model

This notebook assumes you have uploaded the 2 zip files:
- `Lending_default_train.zip`
- `Lending_default_holdout.zip`

In [ ]:
# Install required libraries
!pip install -q pandas numpy scikit-learn optbinning probatus shap matplotlib seaborn statsmodels -U

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, balanced_accuracy_score, precision_score, recall_score

import shap
shap.initjs()

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('✓ All libraries imported successfully!')

# Step 1: Extract and Load Data

In [ ]:
# Check files in current directory
print('Files in current directory:')
for file in os.listdir('.'):
    if file.endswith(('.zip', '.csv')):
        print(f'  - {file}')

In [ ]:
# Extract zip files
zip_files = ['Lending_default_train.zip', 'Lending_default_holdout.zip']

for zip_file in zip_files:
    if os.path.exists(zip_file):
        print(f'Extracting {zip_file}...')
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall()
        print(f'  ✓ Done')

# List CSV files
csv_files = sorted([f for f in os.listdir('.') if f.endswith('.csv')])
print(f'\nExtracted CSV files ({len(csv_files)}):')  
for f in csv_files:
    print(f'  - {f}')

In [ ]:
# Load data
print('Loading training data...')
df_train_tx = pd.read_csv('Lending_default_train_tx.csv')
df_train_acc = pd.read_csv('Lending_default_train_account.csv')
df_train_label = pd.read_csv('Lending_default_train_label.csv')

print('Loading holdout data...')
df_holdout_tx = pd.read_csv('Lending_default_holdout_tx.csv')
df_holdout_acc = pd.read_csv('Lending_default_holdout_account.csv')

# Remove unnamed columns
for df in [df_train_tx, df_train_acc, df_train_label, df_holdout_tx, df_holdout_acc]:
    cols_to_drop = [col for col in df.columns if col.startswith('Unnamed')]
    df.drop(columns=cols_to_drop, inplace=True)

print('✓ Data loaded successfully!')

# Step 2: Data Overview

In [ ]:
print('='*80)
print('TRAINING DATA OVERVIEW')
print('='*80)
print(f'\nTransaction Data:')
print(f'  Shape: {df_train_tx.shape}')
print(f'  Columns: {list(df_train_tx.columns)}')
print(f'  Sample:')
print(df_train_tx.head(3))

print(f'\nAccount Data:')
print(f'  Shape: {df_train_acc.shape}')
print(f'  Columns: {list(df_train_acc.columns)}')
print(f'  Sample:')
print(df_train_acc.head(3))
print(f'  Missing values:')
print(df_train_acc.isnull().sum())

print(f'\nLabel Data:')
print(f'  Shape: {df_train_label.shape}')
print(f'  Distribution:')
print(df_train_label['loan_default'].value_counts())
event_rate = round(df_train_label['loan_default'].mean()*100, 2)
print(f'  Event Rate: {event_rate}%')

# Step 3: Merge Data into Modeling Dataset

In [ ]:
# Merge training data
print('Creating training modeling dataset...')
df_train_model = df_train_tx.merge(df_train_acc, on='Restaurant_ID', how='inner')
df_train_model = df_train_model.merge(df_train_label, on='Restaurant_ID', how='inner')

# Merge holdout data
print('Creating holdout modeling dataset...')
df_holdout_model = df_holdout_tx.merge(df_holdout_acc, on='Restaurant_ID', how='inner')

print(f'\nDataset shapes:')
print(f'  Training: {df_train_model.shape}')
print(f'  Holdout: {df_holdout_model.shape}')
print(f'\nTotal columns: {df_train_model.shape[1]}')
print(f'Feature columns (excluding target): {df_train_model.shape[1] - 1}')

# Step 4: 80/20 Train-Test Split

In [ ]:
# Prepare features and target
y = df_train_model['loan_default']
drop_cols = ['loan_default', 'Restaurant_ID', 'Tx_date']
X = df_train_model.drop(columns=[col for col in drop_cols if col in df_train_model.columns])

print(f'Total samples: {len(X):,}')
print(f'Total features: {X.shape[1]}')

# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Create dataframes for later use
df_train = X_train.copy()
df_train['loan_default'] = y_train.values
df_test = X_test.copy()
df_test['loan_default'] = y_test.values

print('\n' + '='*80)
print('TRAIN-TEST SPLIT')
print('='*80)
print(f'Training set: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test set: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nEvent rates:')
print(f'  Training: {y_train.mean()*100:.2f}%')
print(f'  Test: {y_test.mean()*100:.2f}%')
print(f'  Overall: {y.mean()*100:.2f}%')
print(f'\n✓ Split complete')

# Step 5: Train Models

In [ ]:
# Logistic Regression
print('Training Logistic Regression...')
lr_model = LogisticRegression(n_jobs=-1, random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Predictions
df_train['pred_lr'] = lr_model.predict_proba(X_train)[:, 1]
df_test['pred_lr'] = lr_model.predict_proba(X_test)[:, 1]

# Performance
train_auc_lr = roc_auc_score(y_train, df_train['pred_lr'])
test_auc_lr = roc_auc_score(y_test, df_test['pred_lr'])
print(f'  Train AUC: {train_auc_lr:.4f}')
print(f'  Test AUC: {test_auc_lr:.4f}')
print(f'  ✓ Done')

In [ ]:
# Gradient Boosting
print('Training Gradient Boosting...')
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_leaf_nodes=10,
    learning_rate=0.1,
    min_samples_leaf=0.05,
    random_state=42
)
gb_model.fit(X_train, y_train)

# Predictions
df_train['pred_gb'] = gb_model.predict_proba(X_train)[:, 1]
df_test['pred_gb'] = gb_model.predict_proba(X_test)[:, 1]

# Performance
train_auc_gb = roc_auc_score(y_train, df_train['pred_gb'])
test_auc_gb = roc_auc_score(y_test, df_test['pred_gb'])
print(f'  Train AUC: {train_auc_gb:.4f}')
print(f'  Test AUC: {test_auc_gb:.4f}')
print(f'  ✓ Done')

# Step 6: Model Comparison

In [ ]:
print('='*80)
print('MODEL PERFORMANCE COMPARISON')
print('='*80)

# Create comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Gradient Boosting'],
    'Train AUC': [train_auc_lr, train_auc_gb],
    'Test AUC': [test_auc_lr, test_auc_gb],
    'Overfit Gap': [abs(train_auc_lr - test_auc_lr), abs(train_auc_gb - test_auc_gb)]
})

print('\n' + results.round(4).to_string(index=False))

# Select best
best_model_idx = 0 if test_auc_lr > test_auc_gb else 1
best_model_name = results.iloc[best_model_idx]['Model']
best_auc = results.iloc[best_model_idx]['Test AUC']
best_model = lr_model if best_model_idx == 0 else gb_model
best_pred_col = 'pred_lr' if best_model_idx == 0 else 'pred_gb'

print(f'\n✓ Best Model: {best_model_name} (Test AUC: {best_auc:.4f})')

# Step 7: Performance Metrics

In [ ]:
def print_metrics(y_true, y_pred_prob, set_name='Test'):
    """Print classification metrics"""
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    print(f'\n{set_name} Set Metrics:')
    print(f'  Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print(f'  Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}')
    print(f'  Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'  Recall: {recall_score(y_true, y_pred):.4f}')
    print(f'  ROC-AUC: {roc_auc_score(y_true, y_pred_prob):.4f}')
    
    cm = confusion_matrix(y_true, y_pred)
    print(f'\n  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:,} | FP: {cm[0,1]:,}')
    print(f'    FN: {cm[1,0]:,} | TP: {cm[1,1]:,}')

print('='*80)
print(f'BEST MODEL: {best_model_name}')
print('='*80)

print_metrics(y_train, df_train[best_pred_col], 'Train')
print_metrics(y_test, df_test[best_pred_col], 'Test')

# Step 8: Time Series - Actual vs Predicted by Period

In [ ]:
def plot_calibration(df, pred_col, actual_col, set_name='Test', n_periods=10):
    """Plot actual vs predicted rates by period"""
    df_temp = df.copy()
    df_temp['period'] = pd.qcut(range(len(df_temp)), q=n_periods, 
                                 labels=[f'P{i+1}' for i in range(n_periods)])
    
    # Calculate rates
    ts = df_temp.groupby('period').agg({
        actual_col: ['mean', 'sum', 'count'],
        pred_col: 'mean'
    }).round(4)
    
    ts.columns = ['actual_rate', 'events', 'total', 'pred_rate']
    ts['actual_pct'] = ts['actual_rate'] * 100
    ts['pred_pct'] = ts['pred_rate'] * 100
    ts['deviation'] = abs(ts['actual_pct'] - ts['pred_pct'])
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    x = range(len(ts))
    
    ax.plot(x, ts['actual_pct'], marker='o', linewidth=2.5, markersize=8,
            label='Actual Default Rate', color='#d62728')
    ax.plot(x, ts['pred_pct'], marker='s', linewidth=2.5, markersize=8,
            label='Predicted Default Rate', color='#1f77b4')
    
    ax.set_xlabel('Period', fontsize=12, fontweight='bold')
    ax.set_ylabel('Default Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{set_name} Set: Actual vs Predicted Default Rates by Period',
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ts.index, rotation=0)
    ax.legend(fontsize=11, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Metrics
    mae = ts['deviation'].mean()
    max_dev = ts['deviation'].max()
    
    print(f'\n{set_name} Set Calibration Metrics:')
    print(f'  MAE (Mean Absolute Error): {mae:.2f}%')
    print(f'  Max Deviation: {max_dev:.2f}%')
    print(f'\nPeriod Summary:')
    print(ts[['actual_pct', 'pred_pct', 'deviation', 'total']].round(2).to_string())
    
    return ts

# Plot for training
print('='*80)
print('TRAINING DATA: Actual vs Predicted Default Rates')
print('='*80)
train_calib = plot_calibration(df_train, best_pred_col, 'loan_default', 'Train')

In [ ]:
# Plot for test
print('\n' + '='*80)
print('TEST DATA: Actual vs Predicted Default Rates')
print('='*80)
test_calib = plot_calibration(df_test, best_pred_col, 'loan_default', 'Test')

# Step 9: Score Holdout Data

In [ ]:
# Prepare holdout features
X_holdout = df_holdout_model.drop(
    columns=[col for col in ['Restaurant_ID', 'Tx_date'] if col in df_holdout_model.columns]
)

# Align with training features
for col in X_train.columns:
    if col not in X_holdout.columns:
        X_holdout[col] = 0

X_holdout = X_holdout[X_train.columns]

# Score
holdout_pred = best_model.predict_proba(X_holdout)[:, 1]

# Create results
holdout_results = pd.DataFrame({
    'Restaurant_ID': df_holdout_model['Restaurant_ID'].values,
    'Predicted_Default_Probability': np.round(holdout_pred, 4),
    'Predicted_Default_Score_0_100': np.round(holdout_pred * 100, 2)
})

print('='*80)
print(f'HOLDOUT SCORING - {best_model_name}')
print('='*80)
print(f'Total records: {len(holdout_results):,}')
print(f'\nPrediction Statistics:')
print(holdout_results['Predicted_Default_Probability'].describe().round(4))
print(f'\nFirst 10 predictions:')
print(holdout_results.head(10).to_string(index=False))

# Save
holdout_results.to_csv('holdout_predictions.csv', index=False)
print(f'\n✓ Saved to holdout_predictions.csv')

# Step 10: Holdout Prediction Distribution

In [ ]:
# Holdout distribution by decile
holdout_with_period = holdout_results.copy()
holdout_with_period['decile'] = pd.qcut(holdout_with_period['Predicted_Default_Probability'], 
                                         q=10, labels=[f'D{i+1}' for i in range(10)])

holdout_dist = holdout_with_period.groupby('decile').agg({
    'Predicted_Default_Probability': ['mean', 'std', 'min', 'max'],
    'Restaurant_ID': 'count'
}).round(4)

holdout_dist.columns = ['mean_score', 'std_score', 'min_score', 'max_score', 'count']
holdout_dist['pct_count'] = (holdout_dist['count'] / holdout_dist['count'].sum() * 100).round(1)

print('='*80)
print('HOLDOUT DISTRIBUTION BY DECILE')
print('='*80)
print(holdout_dist.to_string())

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(holdout_results['Predicted_Default_Probability'], bins=50, 
        color='#1f77b4', alpha=0.7, edgecolor='black')
ax.axvline(holdout_results['Predicted_Default_Probability'].mean(), 
           color='#d62728', linestyle='--', linewidth=2, label='Mean')
ax.set_xlabel('Predicted Default Probability', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Holdout Set: Prediction Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Step 11: Summary & Download

In [ ]:
print('='*80)
print('MODELING SUMMARY')
print('='*80)
print(f'\nDataset Split:')
print(f'  Training samples: {len(df_train):,}')
print(f'  Test samples: {len(df_test):,}')
print(f'  Holdout samples: {len(holdout_results):,}')
print(f'\nBest Model: {best_model_name}')
print(f'  Test ROC-AUC: {best_auc:.4f}')
print(f'\nFiles Generated:')
print(f'  - holdout_predictions.csv')
print(f'\nTo Download in Colab:')
print(f'  1. Click the folder icon on the left')
print(f'  2. Right-click holdout_predictions.csv')
print(f'  3. Select Download')
print(f'\n✓ Modeling Complete!')

In [ ]:
# Optional: Download programmatically
# Uncomment the line below to auto-download
# from google.colab import files
# files.download('holdout_predictions.csv')